# HAKE-MER — M1+M2+M3 (SenticNet per-label prior, H3 vs NRC)

**Modules:** M1 + M2 + M3 (SenticNet per-label prior, gate $g$ init 0) · DistilBERT-base

**Protocol:** batch 16, LR 5e-5, 4 epochs, 3 seeds.

Compare test F1-macro to **M1+M2: 0.505 ± 0.004**, **M3 SenticNet (document prior): 0.504 ± 0.003**, and **M3 NRC per-label prior** (`m1_m2_m3_nrc_emotion_specific_campaign.ipynb`).

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
fb555d0


In [3]:
!pip install -q -r requirements-train.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.9 MB/s eta 0:00:00


In [4]:
!./run_m1_m2_m3_emotion_specific_campaign.sh senticnet --backbone distilbert-base-uncased {TRAIN_FLAGS}

config.json: 100% 483/483 [00:00<00:00, 5.14MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 544kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 5.18MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 9.84MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 56.7MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.77M [00:00<?, ?B/s]
simplified/train-00000-of-00001.parquet: downloading bytes:  92% 2.54M/2.77M [00:00<00:00, 4.16MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 3.60MB/s,  265kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 3.65MB/s,  271kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/350k [00:00<?, ?B/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 450kB/s, 33.9kB/s  ]
simplified/validation-00000-of-00001.par(…): reconstructing file: 100% 350k/350k [00:00<00:00, 456kB

In [5]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_m3_senticnet_es_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
if "lexicon_gate_abs" in c:
    print("  |g| mean:", c["lexicon_gate_abs"]["mean"])
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

m1_m2_m3_senticnet_es_distilbert_base_uncased_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_phrases': 4, 'phrase_max_length': 32, 'early_stopping_patience': 0}
  |g| mean: 0.0369097168246905
  f1_micro: 0.5791 ± 0.0020
  f1_macro: 0.5038 ± 0.0007
  exact_match: 0.4556 ± 0.0036
  map: 0.5051 ± 0.0034


In [6]:
import zipfile
from google.colab import files

slug = "distilbert_base_uncased"
campaign = Path(f"reference/artifacts/m1_m2_m3_senticnet_es_{slug}_campaign.json")
out_name = "m1_m2_m3_senticnet_es_distilbert_step0.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_m3_senticnet_es/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download m1_m2_m3_senticnet_es_distilbert_step0.zip (6.9 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Integrity

1. **File → Download → .ipynb** → `reference/training_records/step_m3_es_senticnet/colab/`
2. Save the zip under `reference/training_records/step_m3_es_senticnet/zips/` (or `reference/artifacts/`).
3. Update `reference/training_records/step_m3_es_senticnet/manifest.json` with test aggregates and commit hash.